# Semana 43 · ¿De qué está hecho el universo?

Este cuaderno acompaña a la sesión de proyecto de la semana 43: el perfil exponencial de un disco, la curva de rotación de una galaxia espiral, el halo de materia oscura que hace falta para explicarla y la expansión acelerada. **Los datos de galaxia son simulados**, con valores parecidos a los de una espiral como la Vía Láctea.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Medir la escala de un disco exponencial

Simulamos el brillo de un disco con $h = 3$ kpc medido a distintas distancias, con un 8 % de ruido, y recuperamos $h$ ajustando una recta a $\ln I$ frente a $r$ (semanas 26 y 40).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(43)
h_real, I0 = 3.0, 1000.0
r = np.linspace(1, 15, 30)
I = I0 * np.exp(-r / h_real) * (1 + rng.normal(0, 0.08, r.size))

pendiente, ordenada = np.polyfit(r, np.log(I), 1)
h_med = -1 / pendiente
print(f"escala de longitud medida: h = {h_med:.2f} kpc (real: {h_real})")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(r, I, "o"); a1.plot(r, np.exp(ordenada + pendiente * r), "r")
a1.set_xlabel("r (kpc)"); a1.set_ylabel("brillo"); a1.set_title("Escala lineal")
a2.semilogy(r, I, "o"); a2.semilogy(r, np.exp(ordenada + pendiente * r), "r")
a2.set_xlabel("r (kpc)"); a2.set_title("Escala logarítmica: una recta")
plt.tight_layout(); plt.show()

## 2. La curva de rotación de lo que se ve

Una galaxia de juguete con un **bulbo** de $1{,}5\times10^{10}$ masas solares y un **disco** exponencial de $5\times10^{10}$ masas solares con $h = 3$ kpc. Para simplificar, calculamos la velocidad circular como si la masa encerrada en cada radio estuviera en una esfera, $v = \sqrt{G\,M(<r)/r}$ (para un disco real la fórmula exacta es algo distinta, pero la forma es parecida).

- Masa del disco dentro de $r$: $M_d\,[1 - (1 + r/h)\,e^{-r/h}]$.
- Masa del bulbo dentro de $r$, con un perfil de Hernquist de escala $a$: $M_b\,r^2/(r + a)^2$.

In [ ]:
G = 4.30091e-6            # constante de la gravitación en kpc · (km/s)² / masa solar

def M_disco(r, Md=5e10, h=3.0):
    return Md * (1 - (1 + r / h) * np.exp(-r / h))

def M_bulbo(r, Mb=1.5e10, a=0.6):
    return Mb * r**2 / (r + a) ** 2

def v_circular(M_encerrada, r):
    return np.sqrt(G * M_encerrada / r)

rr = np.linspace(0.2, 30, 300)
v_visible = v_circular(M_disco(rr) + M_bulbo(rr), rr)
plt.figure(figsize=(9, 5))
plt.plot(rr, v_circular(M_bulbo(rr), rr), "--", label="bulbo")
plt.plot(rr, v_circular(M_disco(rr), rr), "--", label="disco")
plt.plot(rr, v_visible, "k", lw=2, label="bulbo + disco (lo visible)")
plt.xlabel("r (kpc)"); plt.ylabel("velocidad de rotación (km/s)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Curva de rotación si solo existiera lo que brilla"); plt.show()

Más allá de unas pocas escalas de longitud, la curva baja: casi toda la masa visible queda dentro, y las estrellas de fuera deberían girar cada vez más despacio, como los planetas exteriores del Sistema Solar.

## 3. Lo que se observa y el halo que hace falta

Ahora simulamos unas medidas de la velocidad de rotación como las que se obtienen con el hidrógeno a 21 cm: casi planas, alrededor de 200-210 km/s, hasta 30 kpc, con errores de 10 km/s. Añadimos un **halo** esférico de materia oscura con densidad $\rho(r) = \rho_0/[1 + (r/r_c)^2]$, cuya masa encerrada es

$$M_h(<r) = 4\pi\rho_0 r_c^2\,\left[r - r_c\arctan(r/r_c)\right],$$

y buscamos los valores de $\rho_0$ y $r_c$ que minimizan el $\chi^2$ (semana 41) probando una cuadrícula de valores.

In [ ]:
def M_halo(r, rho0, rc):
    return 4 * np.pi * rho0 * rc**2 * (r - rc * np.arctan(r / rc))

def v_total(r, rho0, rc):
    return v_circular(M_disco(r) + M_bulbo(r) + M_halo(r, rho0, rc), r)

# «observaciones» simuladas, generadas con un halo concreto que el ajuste no conoce
r_obs = np.linspace(2, 30, 20)
v_obs = v_total(r_obs, 2.5e7, 6.0) + rng.normal(0, 10, r_obs.size)
sigma_v = 10.0

rhos = np.logspace(6, 8.5, 120)
rcs = np.linspace(1, 15, 120)
chi2 = np.array([[(((v_obs - v_total(r_obs, rho, rc)) / sigma_v) ** 2).sum() for rc in rcs] for rho in rhos])
i, j = np.unravel_index(np.argmin(chi2), chi2.shape)
rho_mejor, rc_mejor = rhos[i], rcs[j]
print(f"mejor ajuste: ρ0 = {rho_mejor:.2e} M☉/kpc³, rc = {rc_mejor:.1f} kpc, χ² = {chi2[i, j]:.1f} con {r_obs.size - 2} grados de libertad")

plt.figure(figsize=(9, 5.5))
plt.errorbar(r_obs, v_obs, yerr=sigma_v, fmt="o", label="«observaciones» simuladas")
plt.plot(rr, v_visible, "k--", label="solo materia visible")
plt.plot(rr, v_circular(M_halo(rr, rho_mejor, rc_mejor), rr), ":", label="solo el halo")
plt.plot(rr, v_total(rr, rho_mejor, rc_mejor), "r", lw=2, label="visible + halo")
plt.xlabel("r (kpc)"); plt.ylabel("v (km/s)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

for R in [10, 30]:
    vis = M_disco(R) + M_bulbo(R)
    oscura = M_halo(R, rho_mejor, rc_mejor)
    print(f"dentro de {R} kpc: visible {vis:.2e} M☉, oscura {oscura:.2e} M☉ → la oscura es {oscura / vis:.1f} veces la visible")

Dentro de la parte visible, el halo y las estrellas se reparten la masa; más lejos, el halo domina cada vez más, y su masa sigue creciendo con el radio (en este modelo, casi en proporción a $r$, que es lo que da una curva plana). Los halos reales se extienden mucho más allá de donde llegan las medidas.

## 4. La composición del universo

In [ ]:
partes = {"energía oscura": 68.5, "materia oscura": 26.6, "materia normal": 4.9}
plt.figure(figsize=(5.5, 5.5))
plt.pie(partes.values(), labels=[f"{k}\n{v} %" for k, v in partes.items()],
        colors=["#4c72b0", "#8172b2", "#dd8452"], startangle=90, wedgeprops={"edgecolor": "white"})
plt.title("Contenido del universo hoy (Planck 2018, redondeado)"); plt.show()

## 5. Frenar o acelerar

Un modelo sencillo de cómo crece el factor de escala $a(t)$ hacia el futuro, partiendo de hoy ($a = 1$). La ecuación que gobierna la expansión (la **ecuación de Friedmann**, para un universo plano) es

$$H(a) = H_0\sqrt{\Omega_m\,a^{-3} + \Omega_\Lambda},$$

donde $\Omega_m$ y $\Omega_\Lambda$ son las fracciones de materia y de energía oscura hoy. La materia se diluye con el volumen ($a^{-3}$); la constante cosmológica, no. La integramos con pasos pequeños (método de Euler, semana 11): en cada paso, $a$ crece $a\,H(a)\,\Delta t$.

In [ ]:
H0 = 67.4 / 3.0857e19 * 3.156e16          # 67,4 km/s/Mpc en 1/(miles de millones de años)
def futuro(Om, OL, t_max=60, dt=0.01):
    a, t, at = 1.0, 0.0, [(0.0, 1.0)]
    while t < t_max:
        a += a * H0 * np.sqrt(Om * a**-3 + OL) * dt
        t += dt
        at.append((t, a))
    return np.array(at)

plt.figure(figsize=(9, 5))
for Om, OL, nombre in [(0.315, 0.685, "nuestro universo (Ωm = 0,315, ΩΛ = 0,685)"), (1.0, 0.0, "solo materia (Ωm = 1)")]:
    t_a = futuro(Om, OL)
    plt.plot(t_a[:, 0], t_a[:, 1], label=nombre)
plt.yscale("log"); plt.xlabel("miles de millones de años desde hoy"); plt.ylabel("factor de escala a")
plt.legend(); plt.grid(alpha=0.3, which="both"); plt.show()
t_a = futuro(0.315, 0.685)
print(f"dentro de 50 000 millones de años, el universo será {t_a[np.argmin(abs(t_a[:, 0] - 50)), 1]:.0f} veces mayor que hoy")

Con energía oscura, a largo plazo la curva es una recta en escala logarítmica: crecimiento exponencial. Sin ella, la expansión se frena y el universo crece cada vez más despacio.

## 6. Experimenta

1. Quita el bulbo de la galaxia de juguete. ¿Qué parte de la curva de rotación cambia?
2. Ajusta las «observaciones» sin halo, cambiando solo la masa del disco. ¿Puedes conseguir una curva plana hasta 30 kpc con un disco más masivo?
3. En la sección 5, prueba un universo con $\Omega_m = 0{,}3$ y $\Omega_\Lambda = 0$, y otro con $\Omega_m = 0{,}05$ y $\Omega_\Lambda = 0{,}95$.

In [ ]:
# Tu experimento aquí